# บทที่ 1: บทนำและการวิเคราะห์ประสิทธิภาพ (Algorithm Analysis)

> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)
> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*
> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)

---

## 1.1 Data Structure และ Algorithm คืออะไร

- **Data Structure** = วิธีการ *จัดเก็บ* และ *จัดระเบียบ* ข้อมูลในหน่วยความจำ เพื่อให้เข้าถึง/แก้ไขได้อย่างมีประสิทธิภาพ
- **Algorithm** = ลำดับขั้นตอนที่ชัดเจนและสิ้นสุดได้ (finite) สำหรับแก้ปัญหาหนึ่ง ๆ
- การเลือกโครงสร้างข้อมูลผิด ทำให้อัลกอริทึมที่ดีก็ยังช้า → ทั้งสองเรื่องต้องเรียนคู่กัน

**เกณฑ์วัดคุณภาพอัลกอริทึม** มี 2 ด้านหลัก
1. **Time complexity** – ใช้เวลา (จำนวนขั้นตอน) เท่าใดเมื่อ input ใหญ่ขึ้น
2. **Space complexity** – ใช้หน่วยความจำเพิ่มเท่าใด

## 1.2 Asymptotic Notation

| สัญกรณ์ | ความหมาย | ใช้พูดถึง |
|---|---|---|
| \(O(f(n))\) | ขอบเขตบน (upper bound) | กรณีแย่สุด (worst case) |
| \(\Omega(f(n))\) | ขอบเขตล่าง (lower bound) | กรณีดีสุด (best case) |
| \(\Theta(f(n))\) | ขอบเขตแน่น (tight bound) | โตในอัตราเดียวกันทั้งบน–ล่าง |

**ลำดับอัตราการเติบโต** (จากดีไปแย่):
\[ O(1) < O(\log n) < O(n) < O(n\log n) < O(n^2) < O(n^3) < O(2^n) < O(n!) \]

In [ ]:
# ทดลอง: เปรียบเทียบอัตราการเติบโตของฟังก์ชันต่าง ๆ (growth rate)
import math

def show_growth(sizes=(10, 100, 1000, 10000, 100000)):
    print(f"{'n':>8} | {'log n':>8} | {'n':>10} | {'n log n':>12} | {'n^2':>14}")
    print("-" * 62)
    for n in sizes:
        print(f"{n:>8} | {math.log2(n):>8.1f} | {n:>10} | {n*math.log2(n):>12.0f} | {n**2:>14}")

show_growth()

In [ ]:
# วัดเวลาจริง (empirical) เพื่อยืนยันทฤษฎี: linear search O(n) vs set lookup O(1)
import time, random

def measure(func, *args, repeat=5):
    # คืนค่าเวลาที่ดีที่สุด (best of repeat) หน่วยเป็นไมโครวินาที
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter()
        func(*args)
        best = min(best, time.perf_counter() - t0)
    return best * 1e6

for n in (1_000, 10_000, 100_000):
    data = list(range(n))
    lookup = set(data)
    target = -1  # ค่าที่ไม่มีอยู่ → บังคับให้เป็น worst case
    t_list = measure(lambda: target in data)
    t_set  = measure(lambda: target in lookup)
    print(f"n={n:>7}  list(O(n)) = {t_list:9.1f} us   set(O(1)) = {t_set:6.1f} us")

## 1.3 การวิเคราะห์จากโค้ด (Counting Operations)

หลักการนับแบบง่าย
- คำสั่งพื้นฐาน (assignment, เปรียบเทียบ, คำนวณ) = \(O(1)\)
- ลูปเดี่ยวที่วน n รอบ = \(O(n)\)
- ลูปซ้อนลูป = คูณกัน → \(O(n^2)\)
- ลูปที่ตัวแปรคูณ/หารด้วยค่าคงที่ทุกรอบ (`i *= 2`) = \(O(\log n)\)
- ลำดับบล็อกโค้ดต่อกัน = เอา **ตัวที่โตเร็วสุด**

In [ ]:
# ตัวอย่างการนับจำนวนครั้งที่ทำงานจริง เพื่อเทียบกับผลวิเคราะห์
def f_linear(n):
    c = 0
    for i in range(n):
        c += 1                 # O(n)
    return c

def f_quadratic(n):
    c = 0
    for i in range(n):
        for j in range(n):
            c += 1             # O(n^2)
    return c

def f_triangular(n):
    c = 0
    for i in range(n):
        for j in range(i, n):  # ลูปในไม่ได้วนครบ n → n(n+1)/2 ยังเป็น O(n^2)
            c += 1
    return c

def f_log(n):
    c, i = 0, 1
    while i < n:
        c += 1
        i *= 2                 # O(log n)
    return c

for n in (8, 16, 32):
    print(f"n={n:>3}  linear={f_linear(n):>4}  quad={f_quadratic(n):>5}  tri={f_triangular(n):>5}  log={f_log(n):>3}")

: 

## 1.4 Recurrence Relation และ Master Theorem

อัลกอริทึมแบบ divide and conquer มักเขียนเป็นสมการเวียนเกิด (recurrence) เช่น
\[ T(n) = a\,T(n/b) + f(n) \]

**Master Theorem** เทียบ \(f(n)\) กับ \(n^{\log_b a}\)
1. ถ้า \(f(n) = O(n^{\log_b a - \epsilon})\) → \(T(n) = \Theta(n^{\log_b a})\)
2. ถ้า \(f(n) = \Theta(n^{\log_b a})\) → \(T(n) = \Theta(n^{\log_b a}\log n)\)
3. ถ้า \(f(n) = \Omega(n^{\log_b a + \epsilon})\) → \(T(n) = \Theta(f(n))\)

ตัวอย่าง
- Merge sort: \(T(n)=2T(n/2)+O(n) \Rightarrow \Theta(n\log n)\) (กรณี 2)
- Binary search: \(T(n)=T(n/2)+O(1) \Rightarrow \Theta(\log n)\)

---
## แบบฝึกหัดบทที่ 1

1. หา time complexity ของโค้ดนี้
```python
c = 0
for i in range(n):
    j = 1
    while j < n:
        c += 1
        j *= 3
```
2. เขียนฟังก์ชัน `count_pairs(arr, k)` นับจำนวนคู่ที่ผลรวมเท่ากับ `k` ให้ได้ \(O(n)\)
3. `T(n) = 3T(n/2) + n` มีคำตอบเป็นอะไร (ใช้ Master Theorem)
4. ฟังก์ชันเวียนเกิดหา Fibonacci แบบตรง ๆ มี complexity เท่าใด และปรับให้เป็น \(O(n)\) อย่างไร

### เฉลยบทที่ 1

In [ ]:
# ข้อ 1: ลูปนอก n รอบ, ลูปใน log_3(n) รอบ  ->  O(n log n)

# ข้อ 2: ใช้ hash set จำค่าที่เจอแล้ว -> O(n) time, O(n) space
def count_pairs(arr, k):
    seen, count = {}, 0
    for x in arr:
        count += seen.get(k - x, 0)     # จับคู่กับค่าที่เคยเจอ
        seen[x] = seen.get(x, 0) + 1
    return count

print(count_pairs([1, 5, 7, -1, 5], 6))   # -> 3 : (1,5) (7,-1) (1,5)

# ข้อ 3: a=3, b=2 -> n^(log2 3) = n^1.585 ; f(n)=n โตช้ากว่า -> กรณี 1 -> Theta(n^log2 3)

# ข้อ 4: recursive ธรรมดาเป็น O(2^n) เพราะคำนวณซ้ำ; ใช้ memoization/iterative -> O(n)
def fib_slow(n):
    return n if n < 2 else fib_slow(n-1) + fib_slow(n-2)      # O(2^n)

def fib_fast(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b                                        # O(n) time, O(1) space
    return a

print([fib_fast(i) for i in range(10)])